In [1]:
!pip install sktime
!pip install dtaidistance


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 103.6 MB/s eta 0:00:00


In [2]:
# --- HYPERPARAMETERS ---

# DATA PATH
NAME = "ElectricDevices"
#NAME = "SharePriceIncrease"
#NAME = "WordSynonyms"
TRAIN_FILENAME        = f"{NAME}_TRAIN.ts"                    # path with the training dataset
TEST_FILENAME         = f"{NAME}_TEST.ts"                     # path with the test dataset
SAVING_PATH           = f"{NAME}_best_model_class.pth"        # path to save the trained WHEN model (as .pth file)
BASELINE_CSV_FILENAME = f"{NAME}_baseline_model_results.csv"  # path to save a .csv file with baselines from ts-learn library
OUTPUT_CSV_FILENAME   = f"{NAME}_full_model_result.csv"       # path to save a .csv file with predictions
COMPARISON_TABLE_PATH = f"{NAME}_comparison_table.png"        # path to save the .csv above as a formatted table
TRAIN_DIST_MATRIX_PATH= f"{NAME}_distance_matrix_trian.csv"
TEST_DIST_MATRIX_PATH = f"{NAME}_distance_matrix_test.csv"

# TRAINING PARAMETERS
TRAIN_PROPORTION = -1
TEST_PROPORTION = -1

# WAVEATT MODULE PARAMETERS
DIM_S = 32    # half of feature number of the LSTM (Ks/2 in the paper)
L     = 7     # as in the paper
EPS   = 1e-7  # as in the paper

# DTWATT MODULE PARAMETERS
DIM_V = 32    # input size of the DtwAtt module
D     = 3     # DTW path window
HEADS = 5     # number of universal feature sequences to learn

# WHEN TRAINING PARAMETERS
PATIENCE = 7  # if no improvement in the validation error after PATIENCE epoch stop training

# UNMODIFIABLE
GAMMA = 8     # number of wavelets families used (DO NOT MODIFY)

def get_model_params():
  reduced_seq = (I - 2*L) - 6
  params = {
      "Kx": DIM_X,
      "Ks": DIM_S,
      "L": L,
      "epsilon": EPS,
      "gamma_size": GAMMA,
      "Kv": DIM_V,
      "LD": D,
      "num_heads": HEADS,
      "seq_len": reduced_seq,
      "num_classes": LABELS
  }
  return params


In [3]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sktime.datasets import load_from_tsfile
from sktime.datatypes._panel._convert import from_nested_to_3d_numpy
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Data Loading
X_train_raw, y_train_raw = load_from_tsfile(TRAIN_FILENAME)
X_test_raw, y_test_raw = load_from_tsfile(TEST_FILENAME)

# 2. Preprocess X
if(NAME == "Handwriting"):
  X_train_3d = from_nested_to_3d_numpy(X_train_raw).transpose(0, 2, 1)
  X_test_3d = from_nested_to_3d_numpy(X_test_raw).transpose(0, 2, 1)
else:
  X_train_3d = from_nested_to_3d_numpy(X_train_raw).transpose(0, 2, 1).squeeze(-1)
  X_test_3d = from_nested_to_3d_numpy(X_test_raw).transpose(0, 2, 1).squeeze(-1)
#print(f"X_train_3d shape: {X_train_3d.shape}")

scaler = StandardScaler()
X_train_3d = scaler.fit_transform(X_train_3d)
X_test_3d = scaler.transform(X_test_3d)

X_train_3d = np.expand_dims(X_train_3d, axis=-1)
X_test_3d = np.expand_dims(X_test_3d, axis=-1)
#print(f"X_train_3d shape: {X_train_3d.shape}")

# 3. Encoding labels
#print(np.unique(y_train_raw))
le = LabelEncoder()
y_train_idx = le.fit_transform(y_train_raw)
y_test_idx = le.transform(y_test_raw)

# 4. Reduce the Dataset (for large dataset)
if(TRAIN_PROPORTION > 0):
  X_train_3d_s, _, y_train_idx_s, _ = train_test_split(
      X_train_3d,
      y_train_idx,
      train_size= TRAIN_PROPORTION,
      stratify= y_train_idx
  )
else:
  X_train_3d_s = X_train_3d
  y_train_idx_s = y_train_idx

if(TEST_PROPORTION > 0):
  _, X_test_3d_s, _, y_test_idx_s = train_test_split(
      X_test_3d,
      y_test_idx,
      test_size = TEST_PROPORTION,
      stratify = y_test_idx
  )
else :
  X_test_3d_s = X_test_3d
  y_test_idx_s = y_test_idx

# 5. Extract dataset features
_, I, DIM_X = X_train_3d.shape
LABELS = int(y_train_idx.max()) + 1

# 4. Check for class Balance
#print('CLASS BALANCE CHECK:')
labels_train, counts_train = np.unique(y_train_idx_s, return_counts=True)
counts_train = counts_train / len(y_train_idx_s)
labels_test, counts_test = np.unique(y_test_idx_s, return_counts = True)
counts_test = counts_test / len(y_test_idx_s)

report = pd.DataFrame({'Train': counts_train, 'Test': counts_test}, index=labels_train)
print(report.T.to_latex())
#display(report)
p_train = np.sum(y_train_idx_s) / len(y_train_idx_s)
p_test = np.sum(y_test_idx_s) / len(y_test_idx_s)

# 5. Conversion to tensor
X_train = torch.tensor(X_train_3d_s, dtype=torch.float32)
y_train = torch.tensor(y_train_idx_s, dtype=torch.long)
X_val = torch.tensor(X_test_3d_s, dtype=torch.float32)
y_val = torch.tensor(y_test_idx_s, dtype=torch.long)

# 6. Dataset and DataLoader creation
train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

#print(f"Train batch X shape: {next(iter(train_loader))[0].shape}")

\begin{tabular}{lrrrrrrr}
\toprule
 & 0 & 1 & 2 & 3 & 4 & 5 & 6 \\
\midrule
Train & 0.081447 & 0.249944 & 0.095339 & 0.165136 & 0.269550 & 0.057024 & 0.081559 \\
Test & 0.086500 & 0.253664 & 0.097912 & 0.151083 & 0.242381 & 0.096356 & 0.072105 \\
\bottomrule
\end{tabular}



In [4]:
## WaveAtt MODULE ##

#INPUT X: [batch, I, Kx]
#OUTPUT R: [batch, I - 2L, Ks = hidden_state * 2, Gamma]

import torch
import torch.nn as nn
import torch.nn.functional as F

class WaveAtt(nn.Module):
    def __init__(self, input_size, hidden_size, L, epsilon=1e-8):
        super(WaveAtt, self).__init__()
        self.L = L
        self.wdw_size = 2 * L + 1
        self.epsilon = epsilon

        # BiLSTM encodes input sequence X into hidden states S (Eq. 3.2)
        self.BiLSTM = nn.LSTM(input_size=input_size,
                              hidden_size=hidden_size,
                              batch_first=True,
                              bidirectional=True)

        # Linear layer to compute the data-dependent dilated scalar alpha (Eq. 4)
        self.alpha_linear = nn.Linear(self.wdw_size, 1, bias=True)
        self.relu = nn.ReLU()

        # Register t as a buffer to handle device movement (CPU/GPU) automatically
        t_range = torch.arange(-self.L, self.L + 1).float()
        self.register_buffer('t', t_range.view(1, 1, 1, -1)) # [1, 1, 1, wdw_size]

    def _wavelet_functions(self, t_scaled):
            pi = torch.acos(torch.zeros(1)).item() * 2

            # 1. Mexican Hat (Ricker)
            psi1 = (1 - t_scaled**2) * torch.exp(-(t_scaled**2) / 2)
            # 2. Morlet (Standard)
            psi2 = torch.cos(5 * t_scaled) * torch.exp(-(t_scaled**2) / 2)
            # 3. Shannon (Sinc)
            psi3 = torch.sinc(t_scaled)
            # 4. Laplacian (1st Deriv Gaussian)
            psi4 = -2 * t_scaled * torch.exp(-t_scaled**2)
            # 5. Sine Packet
            psi5 = torch.sin(2 * pi * t_scaled) * torch.exp(-(t_scaled**2) / 4)
            # 6. Chirp-like
            psi6 = torch.cos(t_scaled**2) * torch.exp(-(t_scaled**2) / 4)
            # 7. High Frequency Morlet
            psi7 = torch.cos(10 * t_scaled) * torch.exp(-(t_scaled**2) / 2)
            # 8. Cauchy Approximation
            psi8 = (1 / (1 + t_scaled**2)) * torch.cos(t_scaled)

            # Stack finale: [batch, I-2L, Ks, wdw_size, 8]
            return torch.stack([
                psi1, psi2, psi3, psi4, psi5, psi6, psi7, psi8
            ], dim=-1)

    def forward(self, X):
        # X: [batch, I, Kx]
        batch, seq_len, Kx = X.shape

        # S: [batch, I, Ks] where Ks = 2 * hidden_size
        S, _ = self.BiLSTM(X)

        # Extract sliding windows: [batch, I - 2L, Ks, wdw_size]
        S_windows = S.unfold(1, self.wdw_size, 1)

        # Compute alpha dynamically: [batch, I - 2L, Ks, 1]
        alpha = self.alpha_linear(S_windows)
        alpha = self.relu(alpha) + self.epsilon

        # Scale time: [batch, I - 2L, Ks, wdw_size]
        t_scaled = self.t / alpha

        # Generate wavelet intensities psi
        psi = self._wavelet_functions(t_scaled)
        psi = psi / torch.sqrt(alpha).unsqueeze(-1)

        # Compute attentions
        psi_sum = torch.sum(torch.abs(psi), dim=-2, keepdim=True)
        attention = psi / (psi_sum + self.epsilon)

        # R: [batch, I - 2L, Ks, Gamma]
        # Unsqueeze S_windows to [batch, I-2L, Ks, wdw_size, 1] to multiply with attention
        R = torch.sum(S_windows.unsqueeze(-1) * attention, dim=-2)

        return R  # R: [batch, I - 2L, Ks, Gamma]

In [5]:
## Task Dependent NN 1 MODULE ##

#INPUT R: [batch, I-2L, Ks, Gamma]
#OUTPUT V: [batch, I -2L - (size_1-1) - (size_2-1), Kv = kernel_2]

class TD_NN_1_Classification(nn.Module):
    def __init__(self, input_dim, kernel_1=128, size_1=3, kernel_2=32, size_2=5):
        """
        input_dim: (Ks * 2 * gamma_size)
        kernel_2: This will be your final latent dimension (Kv)
        """
        super(TD_NN_1_Classification, self).__init__()

        # Conv1
        self.first_conv = nn.Sequential(
            nn.Conv1d(in_channels=input_dim, out_channels=kernel_1, kernel_size=size_1),
            nn.BatchNorm1d(kernel_1),
            nn.ReLU()
        )

        self.second_conv = nn.Sequential(
            nn.Conv1d(in_channels=kernel_1, out_channels=kernel_2, kernel_size=size_2),
            nn.BatchNorm1d(kernel_2),
            nn.ReLU()
        )


    def forward(self, R):
        # R shape: [batch, seq_len, ks_total, gamma]
        batch, seq_len, ks_total, gamma = R.shape

        # 1. Flatten features: [batch, seq_len, features]
        R_flat = R.reshape(batch, seq_len, -1)

        # 2. Transpose for Conv1d: [batch, features, seq_len]
        V = R_flat.transpose(1, 2)
        V = self.first_conv(V)
        V = self.second_conv(V)

        # 4. Transpose back: [batch, new_seq_len, Kv]
        # new_seq_len = seq_len - (size_1 - 1) - (size_2 - 1)
        V = V.transpose(1, 2)

        return V

In [6]:
## DTWAtt MODULE ##

#INPUT V: [batch, I*, kernel_2]
#OUTPUT B: [batch, num_heads, I*-D]

# B: batch
# H: heads of the universial feature sequence
# G: number of warping paths
# Z: maximum lenght of each warping path
# D: window size to compute the warping paths

class DTWAtt(nn.Module):
    def __init__(self, Kv, seq_len, LD=3, num_heads=5):
        super(DTWAtt, self).__init__()
        self.LD = LD
        self.num_heads = num_heads
        self.W = LD + 1

        # U: Universial Feature Sequence
        self.U = nn.Parameter(torch.randn(num_heads, seq_len, Kv))

        # Pre-compute DTW paths list
        paths_list = self._get_warping_paths(self.W, self.W)
        max_z = max(len(p) for p in paths_list)

        p_indices, q_indices = [], []
        for path in paths_list:
            p_coords = [step[0] for step in path] + [path[-1][0]] * (max_z - len(path))
            q_coords = [step[1] for step in path] + [path[-1][1]] * (max_z - len(path))
            p_indices.append(p_coords)
            q_indices.append(q_coords)

        self.register_buffer('idx_p', torch.LongTensor(p_indices))
        self.register_buffer('idx_q', torch.LongTensor(q_indices))

    def _get_warping_paths(self, N, M, n=0, m=0, current_path=None):
        if current_path is None: current_path = [(0, 0)]
        if n == N - 1 and m == M - 1: return [current_path]
        paths = []
        for next_n, next_m in [(n+1, m), (n, m+1), (n+1, m+1)]:
            if next_n < N and next_m < M:
                paths.extend(self._get_warping_paths(N, M, next_n, next_m, current_path + [(next_n, next_m)]))
        return paths

    def forward(self, V):
        # V: [batch, I*, Kv]
        batch, seq_len, Kv = V.shape

        # Unfold both V and U to get local windows
        V_win = V.unfold(1, self.W, 1).transpose(-1, -2) # [B, I* - D, W, Kv]
        U_win = self.U.unfold(1, self.W, 1).transpose(-1, -2) # [H, I* - D, W, Kv]

        # Extract path elements: [Batch, I_new, G, Z, Kv]
        V_path = V_win[:, :, self.idx_p, :].unsqueeze(1) # [B, 1, I*-D, G, Z, Kv]
        U_path = U_win[:, :, self.idx_q, :].unsqueeze(0) # [1, H, I*-D, G, Z, Kv]

        # Calculate Soft-DTW distance
        # square L2-norm
        dist = torch.sum((V_path - U_path)**2, dim=-1) # [B, H, I_new, G, Z]
        d_matrix = torch.sum(dist, dim=-1) # [B, H, I*-D, G]

        # Compute Attention
        att_weights = F.softmax(-d_matrix, dim=-1) # [B, H, I*-D, G]

        # Compute B
        B = torch.sum(att_weights * d_matrix, dim=-1) # [B, H, I*-D]

        return B #[B, H, I*-D]

In [7]:
## DTWAtt MODULE ##

#INPUT V: [batch, I*, Kv]
#      B: [batch, num_heads, I*-D]
#OUTPUT: [batch, num_classes]
import torch
import torch.nn as nn
import torch.nn.functional as F

class TD_NN_2_Classification(nn.Module):
    def __init__(self, num_heads, Kv,num_classes, kernel=16, size=3):
        super(TD_NN_2_Classification, self).__init__()

        self.treat_B = nn.Sequential(
            nn.Conv1d(in_channels=num_heads, out_channels=kernel, kernel_size=size),
            nn.BatchNorm1d(kernel),
            nn.AdaptiveAvgPool1d(1)
        )

        self.treat_V = nn.Sequential(
            nn.Conv1d(in_channels=Kv, out_channels=kernel, kernel_size=size),
            nn.BatchNorm1d(kernel),
            nn.AdaptiveAvgPool1d(1)
        )

        self.fc = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, B, V):

        V_feat = self.treat_V(V.transpose(1, 2)).squeeze(-1)
        B_feat = self.treat_B(B).squeeze(-1)

        combined = torch.cat((B_feat, V_feat), dim=-1) # [Batch, 32]
        output = self.fc(combined)

        return output

In [8]:
# [English comments]
class WHEN_Model_Classification(nn.Module):
    def __init__(self, Kx, Ks, L, epsilon, gamma_size, Kv, LD, num_heads, seq_len, num_classes):
        super(WHEN_Model_Classification, self).__init__()

        # 1. Wavelet Attention: Frequency extraction
        self.wave_att = WaveAtt(input_size=Kx, hidden_size=Ks, L=L, epsilon=epsilon)

        # 2. TD-NN-1: Feature transformation into latent space Kv
        # input_dim: (BiLSTM_hidden * 2 * num_wavelets)
        self.td_nn_1 = TD_NN_1_Classification(input_dim=Ks*2 * gamma_size, kernel_2=Kv)

        # 3. DTW Attention: Inter-sequence alignment
        self.dtw_att = DTWAtt(Kv=Kv, seq_len=seq_len, LD=LD, num_heads=num_heads)

        # 4. TD-NN-2: Classification Head
        # We pass Kv to handle the latent vector and num_heads for the attention map
        self.td_nn_2 = TD_NN_2_Classification(num_heads=num_heads, Kv=Kv, num_classes=num_classes)

    def forward(self, X):
        # R: [batch, seq_red, Ks*2, Gamma]
        R = self.wave_att(X)

        # V: [batch, seq_red, Kv]
        V = self.td_nn_1(R)

        # B: [batch, num_heads, seq_red_LD]
        B = self.dtw_att(V)

        # Final prediction: [batch, num_classes]
        prediction = self.td_nn_2(B, V)

        return prediction


In [9]:
### MODEL TRAINING ###

# 1. Define the Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
when_model = WHEN_Model_Classification(**get_model_params()).to(device)

# 2. Upload Pre-Trained Model
try:
  when_model.load_state_dict(torch.load(SAVING_PATH))
  print(f"Loading pretrained model from file: {SAVING_PATH}...")
  print(f"Training using a pretrained model")
except:
  print("No pretrained model found, starting training from scratch")

# 3. Define Optimizer and Loss function
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(when_model.parameters(), lr=0.001)

# 4. Training loop with patience mechanism
patience = 7
best_val_loss = float('inf')
stop_counter = 0

for epoch in range(100):
    when_model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    # --- Training ---
    for x, y in train_loader:
        x, y = x.to(device), y.to(device).long()
        optimizer.zero_grad()
        pred = when_model(x)

        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        # Compute the loss
        train_loss += loss.item()

        # Compute accuracy
        predicted_class = np.argmax(pred.detach().numpy(), axis=-1)
        correct_train += (predicted_class == y).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_acc = correct_train / len(train_loader.dataset)

    # --- Validation ---
    when_model.eval()
    current_val_loss = 0.0
    correct_val = 0

    with torch.no_grad():
        for vx, vy in val_loader:
            vx, vy = vx.to(device), vy.to(device).long()
            v_pred = when_model(vx)

            v_loss = criterion(v_pred, vy)
            current_val_loss += v_loss.item()

            v_predicted_classes = np.argmax(v_pred.detach().numpy(), axis = -1)
            correct_val += (v_predicted_classes == vy).sum().item()

    avg_val_loss = current_val_loss / len(val_loader)
    val_acc = correct_val / len(val_loader.dataset)

    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.2%}")
    print(f"         | Val Loss:   {avg_val_loss:.4f} | Acc: {val_acc:.2%}")

    # --- Early Stopping su Val Loss ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(when_model.state_dict(), SAVING_PATH)
        stop_counter = 0
        print("--> 🟢 Model improved")

    else:
        stop_counter += 1
        print(f"--> 🔴 No improvement ({stop_counter})")

    if stop_counter >= patience:
        print("EARLY STOPPING: Fine del training.")
        break
from google.colab import files
files.download(SAVING_PATH)


Loading pretrained model from file: ElectricDevices_best_model_class.pth...
Training using a pretrained model


KeyboardInterrupt: 

In [ ]:
import time
import warnings
import torch
import numpy as np
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt

# Distance and metrics
from dtaidistance import dtw
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Sktime imports
from sktime.datasets import load_from_tsfile
from sktime.datatypes._panel._convert import from_nested_to_3d_numpy
from sktime.classification.kernel_based import RocketClassifier

# Sklearn imports
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# PyTorch imports
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
print("Loading data...")
X_train_raw, y_train_raw = load_from_tsfile(TRAIN_FILENAME)
X_test_raw, y_test_raw = load_from_tsfile(TEST_FILENAME)

le = LabelEncoder()
y_train_idx = le.fit_transform(y_train_raw)
y_test_idx = le.transform(y_test_raw)
LABELS = np.max(y_train_idx) + 1
print(f"Number of classes: {LABELS}")

X_train_3d = from_nested_to_3d_numpy(X_train_raw).transpose(0, 2, 1).squeeze(-1)
X_test_3d = from_nested_to_3d_numpy(X_test_raw).transpose(0, 2, 1).squeeze(-1)

if X_train_3d.ndim == 3 and X_train_3d.shape[2] == 1:
    X_train_3d = X_train_3d.squeeze(-1)
    X_test_3d = X_test_3d.squeeze(-1)

scaler = StandardScaler()
X_train_3d_scaled = scaler.fit_transform(X_train_3d)
X_test_3d_scaled = scaler.transform(X_test_3d)

# Strutture dati globali per memorizzare risultati e matrici
results_list = []
confusion_matrices = {}

def evaluate_and_log(y_true, y_pred, model_name, inference_time):
    """Computes metrics, stores the confusion matrix, and logs results."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred,average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred,average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred,average='macro', zero_division=0)

    results_list.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "Time (s)": inference_time
    })

    # Salva la matrice di confusione multiclasse N x N
    cf = confusion_matrix(y_true, y_pred)
    confusion_matrices[model_name] = cf

    print(f"Completed: {model_name} in {inference_time:.2f}s")


# ==========================================
# 2. ROCKET (Baseline)
# ==========================================
print("\nStarting ROCKET training...")
try:
    rocket_model = RocketClassifier(num_kernels=2000)
    start_time = time.time()
    rocket_model.fit(X_train_raw, y_train_idx)
    y_pred_rocket = rocket_model.predict(X_test_raw)
    elapsed = time.time() - start_time
    evaluate_and_log(y_test_idx, y_pred_rocket, "ROCKET", elapsed)
except Exception as e:
    print(f"Error with model ROCKET: {e}")


# ==========================================
# 3. K-NN & SVC WITH PRECOMPUTED DTW KERNEL
# ==========================================
print("\nComputing DTW distance matrices manually...")
start_dtw = time.time()

X_train_db = X_train_3d_scaled.astype(np.double)
X_test_db = X_test_3d_scaled.astype(np.double)

n_train = len(X_train_db)
n_test = len(X_test_db)

# 3.1 Train-Train Distance Matrix
dist_train = dtw.distance_matrix_fast(X_train_db)
dist_train[np.isinf(dist_train)] = 0.0
dist_train = dist_train + dist_train.T - np.diag(np.diag(dist_train))

# 3.2 Test-Train Distance Matrix
dist_test = np.zeros((n_test, n_train), dtype=np.double)
for i in range(n_test):
    for j in range(n_train):
        dist_test[i, j] = dtw.distance_fast(X_test_db[i], X_train_db[j])

print(f"DTW matrices computed safely in {time.time() - start_dtw:.2f}s")

# --- K-NN Evaluation ---
knn_model = KNeighborsClassifier(n_neighbors=1, metric='precomputed')
start_time = time.time()
knn_model.fit(dist_train, y_train_idx)
y_pred_knn = knn_model.predict(dist_test)
elapsed_knn = time.time() - start_time
evaluate_and_log(y_test_idx, y_pred_knn, "K-NN", elapsed_knn)

# --- SVC Evaluation (RBF DTW Kernel) ---
median_dist = np.median(dist_train)
gamma = 1.0 / median_dist if median_dist != 0 else 1.0

kernel_train = np.exp(-gamma * (dist_train ** 2))
kernel_test = np.exp(-gamma * (dist_test ** 2))

svc_model = SVC(kernel='precomputed', C=1.0)
start_time = time.time()
svc_model.fit(kernel_train, y_train_idx)
y_pred_svc = svc_model.predict(kernel_test)
elapsed_svc = time.time() - start_time
evaluate_and_log(y_test_idx, y_pred_svc, "SVC", elapsed_svc)


# ==========================================
# 4. RANDOM FOREST (Rolling Lags)
# ==========================================
print("\nExtracting rolling lag features for Random Forest...")

def extract_rolling_lag_features(X, window_size=5):
    df_features = []
    num_samples = X.shape[0]
    for i in range(num_samples):
        series = pd.Series(X[i])
        features = list(X[i])
        rolling_mean = series.rolling(window=window_size, min_periods=1).mean().values
        rolling_std = series.rolling(window=window_size, min_periods=1).std().fillna(0).values
        lag_diff = series.diff().fillna(0).values
        sample_features = np.concatenate([features, rolling_mean, rolling_std, lag_diff])
        df_features.append(sample_features)
    return np.array(df_features)

window_param = 8
X_train_tabular = extract_rolling_lag_features(X_train_3d_scaled, window_size=window_param)
X_test_tabular = extract_rolling_lag_features(X_test_3d_scaled, window_size=window_param)

rf_model = RandomForestClassifier(
    n_estimators=200,
    criterion="entropy",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
rf_model.fit(X_train_tabular, y_train_idx)
y_pred_rf = rf_model.predict(X_test_tabular)
elapsed_rf = time.time() - start_time
evaluate_and_log(y_test_idx, y_pred_rf, "Random Forest", elapsed_rf)


# ==========================================
# 5. WHEN ARCHITECTURE (PyTorch)
# ==========================================
print("\nEvaluating WHEN Architecture...")
try:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    when_test = WHEN_Model_Classification(**get_model_params()).to(device)
    when_test.load_state_dict(torch.load(SAVING_PATH, map_location=device))
    when_test.eval()

    X_val = torch.tensor(X_test_3d_scaled, dtype=torch.float32).unsqueeze(-1)
    y_val = torch.tensor(y_test_idx, dtype=torch.long)

    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False)

    y_pred_when = []
    start_time = time.time()

    with torch.no_grad():
        for x, _ in val_loader:
            x = x.to(device)
            out = when_test(x)
            preds = torch.argmax(out, dim=-1).cpu().numpy()
            y_pred_when.extend(preds)

    elapsed_when = time.time() - start_time
    evaluate_and_log(y_test_idx, y_pred_when, "WHEN", elapsed_when)

    total_params = sum(p.numel() for p in when_test.parameters() if p.requires_grad)
    print(f"Total Trainable Params (WHEN): {total_params:,}")

except Exception as e:
    print(f"Error evaluating WHEN model: {e}")


# ==========================================
# 6. VISUALIZE CONFUSION MATRICES (1x5 GRID)
# ==========================================
print("\nGenerating Confusion Matrices Plot...")
num_models = len(confusion_matrices)

# Crea una figura larga con 1 riga e 'num_models' colonne
fig, axes = plt.subplots(1, num_models, figsize=(5 * num_models, 4))

for ax, (model_name, cf) in zip(axes, confusion_matrices.items()):
    sn.heatmap(cf, cmap='coolwarm', annot=True, fmt='d', ax=ax, cbar=False)
    ax.set_title(model_name, fontsize=12, pad=10)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.title(f"Confusion matrices for dataset {NAME}")
plt.tight_layout()
plt.show()


# ==========================================
# 7. RESULTS AGGREGATION & EXPORT
# ==========================================
print("\nCompiling final performance matrix...")
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(by="F1-Score", ascending=False)

styled_df = results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-Score": "{:.4f}",
    "Time (s)": "{:.2f}"
})

print("\n=== PERFORMANCE MATRIX ===")
display(styled_df)

results_df.to_csv(OUTPUT_CSV_FILENAME, index=False)
try:
    from google.colab import files
    files.download(OUTPUT_CSV_FILENAME)
except ImportError:
    print(f"\nSaved CSV locally at: {OUTPUT_CSV_FILENAME}")

print("\n=== LATEX CODE ===")
print(results_df.to_latex(
    index=False,
    caption="Performance comparison on dataset",
    label="tab:results",
    float_format="%.4f"
))